[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch03/NN_DL_ch03_FraudDetection/NN_DL_ch03_FraudDetection.ipynb)

# NN_DL_ch03_FraudDetection

**Fraud Detection with MLP on Highly Imbalanced Data**

Simulate a realistic fraud detection dataset with extreme class imbalance. Train an MLP with dropout, weight decay, and class-weighted loss. Evaluate with ROC, precision-recall, and confusion matrix.

In [ ]:
!pip install -q torch scikit-learn matplotlib

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Generate Fraud Dataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from sklearn.datasets import fetch_openml
    data = fetch_openml('creditcard', version=1, as_frame=True, parser='auto')
    df = data.frame
    X = df.drop('Class', axis=1).values.astype(np.float32)
    y = df['Class'].astype(int).values
    print(f"Loaded real Credit Card Fraud dataset: {X.shape}")
except Exception:
    print("Using synthetic fraud dataset (make_classification)")
    X, y = make_classification(
        n_samples=50000, n_features=30, n_informative=15,
        n_redundant=5, n_clusters_per_class=3,
        weights=[0.998, 0.002], flip_y=0.01, random_state=42)
    X = X.astype(np.float32)
    print(f"Synthetic dataset: {X.shape}")

print(f"Fraud rate: {y.mean():.5f} ({y.sum()} frauds out of {len(y)})")

scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# MLP with Dropout and Weight Decay
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x)

model = FraudMLP(X_train.shape[1]).to(device)

fraud_rate = y_train.mean()
pos_weight = torch.tensor([(1 - fraud_rate) / max(fraud_rate, 1e-6)]).to(device)
print(f"Positive weight: {pos_weight.item():.1f}")
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train.astype(np.float32)))
test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test.astype(np.float32)))
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=512)

for epoch in range(50):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb).squeeze(), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            y_sc = torch.sigmoid(model(torch.FloatTensor(X_test).to(device))).cpu().numpy().ravel()
        from sklearn.metrics import roc_auc_score
        print(f"Epoch {epoch+1}: AUC = {roc_auc_score(y_test, y_sc):.4f}")

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, auc

model.eval()
with torch.no_grad():
    y_score = torch.sigmoid(model(torch.FloatTensor(X_test).to(device))).cpu().numpy().ravel()

fpr, tpr, thresholds = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color=MAIN_BLUE, lw=2.5, label=f'MLP (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='grey', ls='--', lw=1)
ax.fill_between(fpr, tpr, alpha=0.15, color=MAIN_BLUE)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - Fraud Detection', fontweight='bold')
ax.legend(loc='lower right', fontsize=12)
save_fig('nn_ch03_fraud_roc')

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

j_scores = tpr - fpr
opt_idx = np.argmax(j_scores)
opt_threshold = thresholds[opt_idx]
print(f"Optimal threshold: {opt_threshold:.4f}")

y_pred = (y_score > opt_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Confusion Matrix - Fraud Detection', fontweight='bold')
save_fig('nn_ch03_fraud_confusion')

In [ ]:
# Precision-Recall Curve
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, _ = precision_recall_curve(y_test, y_score)
ap = average_precision_score(y_test, y_score)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(recall, precision, color=MAIN_BLUE, lw=2.5, label=f'MLP (AP = {ap:.3f})')
ax.fill_between(recall, precision, alpha=0.15, color=MAIN_BLUE)
ax.axhline(y_test.mean(), color='grey', ls='--', lw=1, label=f'Baseline ({y_test.mean():.4f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve - Fraud Detection', fontweight='bold')
ax.legend(fontsize=12); ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
save_fig('nn_ch03_fraud_precision_recall')

## Summary

Trained a fraud detection MLP on highly imbalanced data with class-weighted BCE loss, dropout, and weight decay. Evaluated using ROC-AUC, precision-recall, and confusion matrix with an optimized decision threshold.